In [1]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers datasets peft accelerate bitsandbytes


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 93.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 37.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.0/803.0 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:

In [2]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()


In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [4]:
## Config loading
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

DATASET_PATH = (
    "/kaggle/input/slm-fine-tune/"
    "slm_instruction_finetuning_data.jsonl"
)

OUTPUT_DIR = "./Qwen2.5-0.5B-Instruct"

MAX_LENGTH = 512
SEED = 42


In [5]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)}"
    )
    print(
        f"  Total: "
        f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB"
    )


CUDA: True
GPU count: 2
GPU 0: Tesla T4
  Total: 14.56 GB
GPU 1: Tesla T4
  Total: 14.56 GB


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
dataset = load_dataset(
    "json",
    data_files=DATASET_PATH,
)

print(dataset)
print(dataset["train"].column_names)

print(dataset["train"][0])




Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 534
    })
})
['messages']
{'messages': [{'role': 'system', 'content': 'You are an expert Policy agent for Azilen Technologies. You must answer questions strictly based on the provided context. If the question is out of context, respond politely indicating that.'}, {'role': 'user', 'content': 'Context: Purpose: • To assess employee’s performance & potential. • To identify the best performers who are ready for promotion. • For recognizing any retention risk.\n\nQuestion: What is the main purpose of the Yardstick to Success (Y2S) policy?'}, {'role': 'assistant', 'content': 'The Y2S policy is designed to assess employee performance and potential, identify top performers ready for promotion, and recognize any retention risks.'}]}


In [8]:
split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=SEED,
)

train_test = split["train"].train_test_split(
    test_size=0.125,
    seed=SEED,
)

train_dataset = train_test["train"]
eval_dataset = train_test["test"]
test_dataset = split["test"]

print(
    f"Train: {len(train_dataset)} | "
    f"Eval: {len(eval_dataset)} | "
    f"Test: {len(test_dataset)}"
)


Train: 373 | Eval: 54 | Test: 107


In [9]:
def format_chat(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


train_dataset = train_dataset.map(
    format_chat,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    format_chat,
    remove_columns=eval_dataset.column_names,
)

test_dataset = test_dataset.map(
    format_chat,
    remove_columns=test_dataset.column_names,
)


Map:   0%|          | 0/373 [00:00<?, ? examples/s]

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Map:   0%|          | 0/107 [00:00<?, ? examples/s]

In [10]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)



In [11]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
## LoRA setup 
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
# prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

In [13]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/373 [00:00<?, ? examples/s]

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

In [14]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [16]:
training_args = TrainingArguments(
    output_dir="./phi3-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size= 1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,  # Disable bf16 if you’re using fp16
    tf32=False,  # Disable tf32 (only for Ampere or newer GPUs)
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    max_grad_norm=0.3,
    # warmup_ratio=0.03,
    lr_scheduler_type="constant",
    disable_tqdm=False
)


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

trainer.train()



Step,Training Loss
10,2.757651
20,1.835055
30,1.389085
40,1.301368
50,1.222144
60,1.090563
70,1.018786
80,1.075947
90,1.074714
100,1.030728


TrainOutput(global_step=470, training_loss=0.8067604308432721, metrics={'train_runtime': 2319.8445, 'train_samples_per_second': 1.608, 'train_steps_per_second': 0.203, 'total_flos': 1.50395855044608e+16, 'train_loss': 0.8067604308432721, 'epoch': 10.0})

In [20]:

model.save_pretrained("./qwen2.5-finetuned")
tokenizer.save_pretrained("./qwen2.5-finetuned")


('./qwen2.5-finetuned/tokenizer_config.json',
 './qwen2.5-finetuned/chat_template.jinja',
 './qwen2.5-finetuned/tokenizer.json')

In [23]:
## Loading the model for the inference

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "./qwen2.5-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)


model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [26]:
from peft import PeftModel
model = PeftModel.from_pretrained(
    base_model,
    model_path
)

model.eval()

/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [28]:
def generate(prompt):

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return answer

In [29]:
%%time
print(generate("Can you tell me about azilen Technologies"))

Azilen Technologies is an engineering and IT services company based in Turkey. The organization provides consulting, integration, development, and maintenance solutions for various enterprise needs, including cloud infrastructure, digital transformation, big data analytics, artificial intelligence, blockchain technology, machine learning, and other emerging technologies.
CPU times: user 7.57 s, sys: 3.99 ms, total: 7.57 s
Wall time: 7.56 s
